# Prediction Validation — `future_unseen_examples.csv`

This notebook validates the **real prediction pipeline** (the same `PredictionService`
used by the API) against the 100 examples in `src/data/future_unseen_examples.csv`.

**Goal:** build confidence that the model, together with the KNN imputation logic, produces
sensible prices on data it has never seen.

Validations performed:

1. **Full pipeline** — runs every row with all fields present; counts success/failure and
   summarizes the price distribution (`describe()`).
2. **Realistic range** — checks that prices fall between \$50k and \$5M (a plausible range
   for the Seattle area).
3. **Missing-data robustness** — nulls fields on purpose and confirms the API still
   responds (KNN imputation working).
4. **Imputation reasonableness** — compares the prediction with complete data vs. partial
   data and measures the deviation.

> Note: `future_unseen_examples.csv` carries extra columns (`waterfront`, `view`, `grade`, …)
> that the API does **not** consume. We use only the API contract fields:
> the 7 numeric features + `zipcode`.


In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Resolve the project root (this notebook lives in test/)
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "test":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from services.imputer import KNNImputerService, NUMERIC_HOME_FEATURES  # noqa: E402
from services.predictor import PredictionService  # noqa: E402
from utils.loader import load_demographics, load_features, load_model  # noqa: E402

API_FIELDS = NUMERIC_HOME_FEATURES + ["zipcode"]
print("Fields consumed by the API:", API_FIELDS)


Fields consumed by the API: ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'sqft_above', 'sqft_basement', 'zipcode']


## Setup — load the real prediction pipeline

We assemble exactly the same `PredictionService` that the API's `lifespan` builds at
startup: the model, the feature list, the per-zipcode demographics, and the KNN imputer
already fitted on the historical sales data.

In [2]:
model = load_model(str(PROJECT_ROOT / "src/model/model.pkl"))
model_features = load_features(str(PROJECT_ROOT / "src/model/model_features.json"))
demographics = load_demographics(str(PROJECT_ROOT / "src/data/zipcode_demographics.csv"))

imputer = KNNImputerService()
imputer.fit(str(PROJECT_ROOT / "src/data/kc_house_data.csv"))

service = PredictionService(
    model=model,
    model_features=model_features,
    demographics=demographics,
    imputer=imputer,
)
print("PredictionService ready.")


PredictionService ready.


In [3]:
examples = pd.read_csv(
    PROJECT_ROOT / "src/data/future_unseen_examples.csv",
    dtype={"zipcode": str},
)
print(f"Examples loaded: {len(examples)} rows")
examples[API_FIELDS].head()


Examples loaded: 100 rows


,bedrooms,bathrooms,sqft_living,sqft_lot,floors,sqft_above,sqft_basement,zipcode
0,4,1.00,1680,5043,1.5,1680,0,98118
1,3,2.50,2220,6380,1.5,1660,560,98115
2,3,2.25,1630,10962,1.0,1100,530,98030
3,5,2.50,1710,9720,2.0,1710,0,98005
4,2,1.00,850,6370,1.0,850,0,98126


## Validation 1 — Full pipeline (all fields present)

We run every row through `service.predict()`. We count success vs. failure (a failure here
would be, for example, a zipcode missing from the demographics table) and collect the
prices of the successful predictions.

In [4]:
def run_predictions(df: pd.DataFrame) -> tuple[list[float], list[dict]]:
    """Run the pipeline row by row. Returns (ok_prices, failures)."""
    prices, failures = [], []
    for idx, row in df.iterrows():
        payload = {f: row[f] for f in API_FIELDS}
        try:
            price = service.predict(home_data=payload, zipcode=payload["zipcode"])
            prices.append(price)
        except Exception as exc:  # noqa: BLE001
            failures.append({"index": int(idx), "zipcode": payload["zipcode"], "error": str(exc)})
    return prices, failures


prices, failures = run_predictions(examples)

total = len(examples)
n_ok = len(prices)
n_fail = len(failures)
print(f"Total:      {total}")
print(f"Successful: {n_ok}")
print(f"Failed:     {n_fail}")
if failures:
    display(pd.DataFrame(failures))


Total:      100
Successful: 100
Failed:     0


In [5]:
prices_s = pd.Series(prices, name="predicted_price")
prices_s.describe().round(2)


count        100.00
mean      481762.73
std       204251.30
min       184300.00
25%       303766.50
50%       450080.00
75%       580642.50
max      1241796.00
Name: predicted_price, dtype: float64

In [6]:
# describe() formatted as currency for quick reading
desc = prices_s.describe()
print("Distribution of predicted prices (successful):")
for stat in ["min", "25%", "50%", "75%", "max"]:
    print(f"  {stat:>4}: ${desc[stat]:>14,.0f}")
print(f"  mean: ${desc['mean']:>14,.0f}")
print(f"   std: ${desc['std']:>14,.0f}")


Distribution of predicted prices (successful):
   min: $       184,300
   25%: $       303,766
   50%: $       450,080
   75%: $       580,642
   max: $     1,241,796
  mean: $       481,763
   std: $       204,251


## Validation 2 — Realistic range (\$50k – \$5M)

Domain sanity check: any price outside this range is suspicious for residential homes in
the Seattle area and would warrant investigation.

In [7]:
LOW, HIGH = 50_000, 5_000_000

in_range = prices_s.between(LOW, HIGH)
n_in = int(in_range.sum())
n_out = int((~in_range).sum())

print(f"Accepted range: ${LOW:,} – ${HIGH:,}")
print(f"Within range: {n_in}/{len(prices_s)}")
print(f"Out of range: {n_out}")

if n_out:
    print("\nValues out of range:")
    display(prices_s[~in_range])
else:
    print("\n✅ All prices fall within the realistic range.")


Accepted range: $50,000 – $5,000,000
Within range: 100/100
Out of range: 0

✅ All prices fall within the realistic range.


## Validation 3 — Missing-data robustness (KNN imputation)

We simulate the real business scenario: the agent does not have every field. We null
`bathrooms`, `sqft_lot`, and `sqft_basement` on **every** row and confirm the API still
responds — which is exactly what the KNN imputation solves.

In [8]:
missing_examples = examples.copy()
for field in ["bathrooms", "sqft_lot", "sqft_basement"]:
    missing_examples[field] = None

prices_missing, failures_missing = run_predictions(missing_examples)

print(f"Successful with missing data: {len(prices_missing)}/{len(missing_examples)}")
print(f"Failed:                       {len(failures_missing)}")
if failures_missing:
    display(pd.DataFrame(failures_missing))
else:
    print("\n✅ KNN imputation filled the null fields on 100% of the rows.")

pd.Series(prices_missing, name="predicted_price_missing").describe().round(2)


Successful with missing data: 100/100
Failed:                       0

✅ KNN imputation filled the null fields on 100% of the rows.


count        100.00
mean      484137.49
std       219614.13
min       184300.00
25%       301705.00
50%       429045.00
75%       604349.50
max      1241796.00
Name: predicted_price_missing, dtype: float64

## Validation 4 — Imputation reasonableness (complete vs. partial)

For each row we compare the prediction with **complete data** against the prediction with
the 3 nulled fields. The `partial / complete` ratio should stay close to 1 — imputation
should not drastically distort the price. We use the same tolerance as the integration
tests (0.5x – 1.5x).

In [9]:
full_prices = np.array(prices)
part_prices = np.array(prices_missing)

# positional alignment (same order, both ran 100% of the rows)
assert len(full_prices) == len(part_prices), "count mismatch — investigate failures"

ratios = part_prices / full_prices
within = (ratios >= 0.5) & (ratios <= 1.5)

summary = pd.DataFrame({
    "full": full_prices,
    "partial": part_prices,
    "ratio": ratios,
})

print(f"Rows within tolerance (0.5x–1.5x): {int(within.sum())}/{len(ratios)}")
print(f"Ratio  — min: {ratios.min():.3f} | median: {np.median(ratios):.3f} | max: {ratios.max():.3f}")
print(f"Mean absolute deviation: {np.mean(np.abs(ratios - 1)) * 100:.1f}%")
summary.describe().round(3)


Rows within tolerance (0.5x–1.5x): 100/100
Ratio  — min: 0.582 | median: 1.000 | max: 1.387
Mean absolute deviation: 5.7%


,full,partial,ratio
count,100.000,100.000,100.000
mean,481762.728,484137.492,1.001
std,204251.301,219614.126,0.105
min,184300.000,184300.000,0.582
25%,303766.500,301705.000,0.983
50%,450080.000,429045.000,1.000
75%,580642.500,604349.500,1.032
max,1241796.000,1241796.000,1.387


## Single KPI — Imputation accuracy

The validations above look at the whole distribution. To communicate in **one number**, we
use **imputation accuracy**: how close, on average, the prediction with missing data stays
to the prediction with complete data.

$$\text{Accuracy} = 100\% - \text{MAPE}, \quad
\text{MAPE} = \frac{1}{n}\sum \left|\frac{\text{partial} - \text{complete}}{\text{complete}}\right|$$

> **Important (methodological honesty):** the reference here is the prediction with complete
> data, **not** the actual sale price (which does not exist in this CSV). So the KPI measures
> the imputation's *fidelity* to the model, not error against the market. We report the mean
> (sensitive to outliers) and the median (robust) side by side.

In [10]:
ape = np.abs(part_prices - full_prices) / full_prices  # absolute percentage error per home

mape = ape.mean()
medape = np.median(ape)

acc_mean = (1 - mape) * 100
acc_median = (1 - medape) * 100

print("KPI — IMPUTATION ACCURACY")
print("=" * 45)
print(f"  Mean accuracy:    {acc_mean:5.1f}%   (mean error {mape * 100:.1f}%)")
print(f"  Median accuracy:  {acc_median:5.1f}%   (median error {medape * 100:.1f}%)")
print("=" * 45)
print(f"\n➡️  HEADLINE: imputation is {acc_mean:.0f}% accurate relative to the complete-data prediction.")


KPI — IMPUTATION ACCURACY
  Mean accuracy:     94.3%   (mean error 5.7%)
  Median accuracy:   98.0%   (median error 2.0%)

➡️  HEADLINE: imputation is 94% accurate relative to the complete-data prediction.


## Summary

| Validation | What it checks | Pass criterion |
|---|---|---|
| 1. Full pipeline | Every row produces a price | 100% success |
| 2. Realistic range | Prices in \$50k–\$5M | 0 out of range |
| 3. Robustness to nulls | API responds with missing fields | 100% success |
| 4. Reasonableness | Imputation does not distort the price | ratio in 0.5x–1.5x |
| **KPI** | **Imputation accuracy** | **~94% (mean error ~6%)** |

The cell below consolidates the verdict.

In [11]:
checks = {
    "1. Full pipeline (100% success)": n_fail == 0,
    "2. Realistic range ($50k-$5M)": n_out == 0,
    "3. Missing-data robustness (100% success)": len(failures_missing) == 0,
    "4. Reasonable imputation (all within 0.5x-1.5x)": bool(within.all()),
    "KPI. Imputation accuracy >= 90%": acc_mean >= 90,
}

print("VALIDATION VERDICT\n" + "=" * 45)
for name, ok in checks.items():
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
print("=" * 45)
print("RESULT:", "ALL PASSED ✅" if all(checks.values()) else "THERE ARE FAILURES ❌")


VALIDATION VERDICT
  [PASS] 1. Full pipeline (100% success)
  [PASS] 2. Realistic range ($50k-$5M)
  [PASS] 3. Missing-data robustness (100% success)
  [PASS] 4. Reasonable imputation (all within 0.5x-1.5x)
  [PASS] KPI. Imputation accuracy >= 90%
RESULT: ALL PASSED ✅
